# 🚛 Ödev: Tool Calling — LojistikAI

**Hedef:** Public API'lerden veri çeken bir tool calling sistemi kurup Hugging Face Spaces'te yayınlamak.

### Araçlar (hepsi ücretsiz, API anahtarsız)
| Araç | Kaynak | Lojistik kullanımı |
|------|--------|--------------------|
| `get_weather` | Open-Meteo | Sevkiyat koşulları |
| `get_route_distance` | OSRM | Rota mesafe & süre |
| `get_exchange_rate` | Frankfurter | Navlun maliyeti |
| `convert_unit` | Yerel hesap | Birim dönüşümü |

### Notebook akışı
1. Kurulum ve API anahtarları
2. Araç fonksiyonlarını yaz ve **tek tek test et**
3. Tool şemalarını (JSON Schema) tanımla
4. Tool calling döngüsünü kur ve test et
5. Gradio arayüzünü Colab'da canlı dene
6. Dosyaları oluştur (`app.py`, `requirements.txt`, `README.md`)
7. **Hugging Face Spaces'e yayınla**

---


## 0. Kurulum

In [1]:
%%capture
!pip install -q gradio openai requests huggingface_hub

In [2]:
import os, json, requests

# ===== API anahtarları =====
# Colab: Sol menü 🔑 Secrets → OPENAI_API_KEY ve HF_TOKEN ekleyin
try:
    from google.colab import userdata
    OPENAI_KEY = userdata.get('OPENAI_API_KEY')
    HF_TOKEN   = userdata.get('HF_TOKEN')
except Exception:
    OPENAI_KEY = input("OpenAI API key: ")
    HF_TOKEN   = input("HuggingFace token: ")

os.environ["OPENAI_API_KEY"] = OPENAI_KEY
os.environ["HF_TOKEN"] = HF_TOKEN

# ===== Ayarlar =====
HF_USERNAME = "cihatyldz"                       # ← kendi kullanıcı adınız
SPACE_NAME  = "lojistik-tool-calling"           # ← Space adı
SPACE_ID    = f"{HF_USERNAME}/{SPACE_NAME}"
MODEL       = "gpt-4o-mini"

print(f"Space  : {SPACE_ID}")
print(f"Model  : {MODEL}")
print("✓ Kurulum tamam")

Space  : cihatyldz/lojistik-tool-calling
Model  : gpt-4o-mini
✓ Kurulum tamam


---
## 1. Araç Fonksiyonları

Her araç bir public API'ye bağlanır. Hepsi ücretsiz ve anahtar gerektirmez.


In [3]:
API_TIMEOUT = 15

def get_weather(city: str) -> dict:
    """Open-Meteo ile şehrin güncel hava durumunu getirir. API anahtarı gerektirmez."""
    try:
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1, "language": "tr", "format": "json"},
            timeout=API_TIMEOUT,
        ).json()

        if not geo.get("results"):
            return {"hata": f"'{city}' şehri bulunamadı"}

        loc = geo["results"][0]
        lat, lon = loc["latitude"], loc["longitude"]

        wx = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": lat,
                "longitude": lon,
                "current": "temperature_2m,precipitation,wind_speed_10m,weather_code",
                "timezone": "auto",
            },
            timeout=API_TIMEOUT,
        ).json()

        cur = wx["current"]
        kod = cur.get("weather_code", 0)
        durum = _weather_code_tr(kod)

        return {
            "sehir": loc["name"],
            "ulke": loc.get("country", ""),
            "sicaklik_c": cur["temperature_2m"],
            "yagis_mm": cur.get("precipitation", 0),
            "ruzgar_kmh": cur.get("wind_speed_10m", 0),
            "durum": durum,
        }
    except Exception as e:
        return {"hata": f"Hava durumu alınamadı: {e}"}


def get_route_distance(origin: str, destination: str) -> dict:
    """OSRM ile iki şehir arası karayolu mesafesi ve tahmini sürüş süresi."""
    try:
        coords = []
        for sehir in (origin, destination):
            geo = requests.get(
                "https://geocoding-api.open-meteo.com/v1/search",
                params={"name": sehir, "count": 1, "language": "tr", "format": "json"},
                timeout=API_TIMEOUT,
            ).json()
            if not geo.get("results"):
                return {"hata": f"'{sehir}' şehri bulunamadı"}
            r = geo["results"][0]
            coords.append((r["longitude"], r["latitude"], r["name"]))

        (lon1, lat1, ad1), (lon2, lat2, ad2) = coords

        route = requests.get(
            f"https://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}",
            params={"overview": "false"},
            timeout=API_TIMEOUT,
        ).json()

        if route.get("code") != "Ok" or not route.get("routes"):
            return {"hata": "Rota hesaplanamadı (karayolu bağlantısı olmayabilir)"}

        r0 = route["routes"][0]
        km = round(r0["distance"] / 1000, 1)
        dk = round(r0["duration"] / 60)

        return {
            "kalkis": ad1,
            "varis": ad2,
            "mesafe_km": km,
            "sure_dakika": dk,
            "sure_okunabilir": f"{dk // 60} sa {dk % 60} dk",
        }
    except Exception as e:
        return {"hata": f"Rota alınamadı: {e}"}


def get_exchange_rate(from_currency: str, to_currency: str, amount: float = 1.0) -> dict:
    """Frankfurter API ile güncel döviz kuru çevrimi. API anahtarı gerektirmez."""
    try:
        f = from_currency.upper()
        t = to_currency.upper()
        data = requests.get(
            "https://api.frankfurter.app/latest",
            params={"from": f, "to": t, "amount": amount},
            timeout=API_TIMEOUT,
        ).json()

        if "rates" not in data or t not in data["rates"]:
            return {"hata": f"Kur bulunamadı: {f} → {t}"}

        return {
            "kaynak_para": f,
            "hedef_para": t,
            "miktar": amount,
            "sonuc": round(data["rates"][t], 4),
            "kur": round(data["rates"][t] / amount, 6),
            "tarih": data.get("date", ""),
        }
    except Exception as e:
        return {"hata": f"Kur alınamadı: {e}"}


def convert_unit(value: float, from_unit: str, to_unit: str) -> dict:
    """Sıcaklık, mesafe ve ağırlık birimleri arasında dönüşüm yapar (yerel hesap)."""
    f = from_unit.upper().strip()
    t = to_unit.upper().strip()

    # Sıcaklık
    if {f, t} <= {"C", "F", "K"}:
        if f == "C":
            c = value
        elif f == "F":
            c = (value - 32) * 5 / 9
        else:
            c = value - 273.15

        sonuc = c if t == "C" else (c * 9 / 5 + 32 if t == "F" else c + 273.15)
        return {"deger": round(sonuc, 2), "birim": t, "kaynak": f"{value} {f}"}

    # Mesafe
    metre = {"KM": 1000, "M": 1, "MIL": 1609.34, "MI": 1609.34, "NM": 1852, "FT": 0.3048}
    if f in metre and t in metre:
        sonuc = value * metre[f] / metre[t]
        return {"deger": round(sonuc, 3), "birim": t, "kaynak": f"{value} {f}"}

    # Ağırlık
    gram = {"KG": 1000, "G": 1, "TON": 1_000_000, "LB": 453.592, "OZ": 28.3495}
    if f in gram and t in gram:
        sonuc = value * gram[f] / gram[t]
        return {"deger": round(sonuc, 3), "birim": t, "kaynak": f"{value} {f}"}

    return {"hata": f"Desteklenmeyen dönüşüm: {f} → {t}"}


def _weather_code_tr(code: int) -> str:
    """WMO hava kodunu Türkçe açıklamaya çevirir."""
    tablo = {
        0: "açık", 1: "az bulutlu", 2: "parçalı bulutlu", 3: "kapalı",
        45: "sisli", 48: "kırağılı sis",
        51: "hafif çisenti", 53: "çisenti", 55: "yoğun çisenti",
        61: "hafif yağmur", 63: "yağmurlu", 65: "şiddetli yağmur",
        71: "hafif kar", 73: "karlı", 75: "yoğun kar",
        80: "sağanak", 81: "kuvvetli sağanak", 82: "şiddetli sağanak",
        95: "gök gürültülü fırtına", 96: "dolulu fırtına",
    }
    return tablo.get(code, "bilinmiyor")

print("✓ 4 araç fonksiyonu tanımlandı")

✓ 4 araç fonksiyonu tanımlandı


### 1.1 Araçları Tek Tek Test Et

In [4]:
# ---- convert_unit (API gerektirmez) ----
print("convert_unit testleri:")
for v, f, t in [(27, 'C', 'F'), (18, 'C', 'F'), (100, 'KM', 'MIL'), (1, 'TON', 'KG')]:
    print(f"  {v}{f} → {t}: {convert_unit(v, f, t)}")

# ---- get_weather ----
print("\nget_weather testi:")
print(f"  {get_weather('İstanbul')}")

# ---- get_route_distance ----
print("\nget_route_distance testi:")
print(f"  {get_route_distance('İstanbul', 'Ankara')}")

# ---- get_exchange_rate ----
print("\nget_exchange_rate testi:")
print(f"  {get_exchange_rate('EUR', 'TRY', 2500)}")

convert_unit testleri:
  27C → F: {'deger': 80.6, 'birim': 'F', 'kaynak': '27 C'}
  18C → F: {'deger': 64.4, 'birim': 'F', 'kaynak': '18 C'}
  100KM → MIL: {'deger': 62.137, 'birim': 'MIL', 'kaynak': '100 KM'}
  1TON → KG: {'deger': 1000.0, 'birim': 'KG', 'kaynak': '1 TON'}

get_weather testi:
  {'sehir': 'İstanbul', 'ulke': 'Türkiye Cumhuriyeti', 'sicaklik_c': 24.4, 'yagis_mm': 0.0, 'ruzgar_kmh': 20.9, 'durum': 'açık'}

get_route_distance testi:
  {'kalkis': 'İstanbul', 'varis': 'Ankara', 'mesafe_km': 445.7, 'sure_dakika': 289, 'sure_okunabilir': '4 sa 49 dk'}

get_exchange_rate testi:
  {'kaynak_para': 'EUR', 'hedef_para': 'TRY', 'miktar': 2500, 'sonuc': 134847, 'kur': 53.9388, 'tarih': '2026-07-29'}


---
## 2. Tool Tanımları (JSON Şeması)

Modelin hangi aracı ne zaman çağıracağını anlaması için her araç
OpenAI function calling formatında tanımlanır. `description` alanı kritik —
model kararını buna göre verir.


In [5]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": (
                "Belirtilen şehrin güncel hava durumunu getirir. Sevkiyat planlaması, "
                "rota güvenliği ve soğuk zincir kararları için kullanılır."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Şehir adı, örn: 'İstanbul', 'Hamburg', 'Rotterdam'",
                    }
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_route_distance",
            "description": (
                "İki şehir arasındaki karayolu mesafesini (km) ve tahmini sürüş süresini "
                "hesaplar. Nakliye maliyeti, teslim süresi ve rota planlaması için kullanılır."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {"type": "string", "description": "Kalkış şehri"},
                    "destination": {"type": "string", "description": "Varış şehri"},
                },
                "required": ["origin", "destination"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_exchange_rate",
            "description": (
                "Güncel döviz kuru ile para birimi çevrimi yapar. Navlun fiyatlandırma, "
                "ithalat/ihracat maliyet hesaplama için kullanılır."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "from_currency": {
                        "type": "string",
                        "description": "Kaynak para birimi kodu, örn: 'USD', 'EUR', 'TRY'",
                    },
                    "to_currency": {
                        "type": "string",
                        "description": "Hedef para birimi kodu, örn: 'TRY', 'USD'",
                    },
                    "amount": {
                        "type": "number",
                        "description": "Çevrilecek miktar (varsayılan 1)",
                    },
                },
                "required": ["from_currency", "to_currency"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "convert_unit",
            "description": (
                "Sıcaklık (C/F/K), mesafe (km/m/mil/nm/ft) ve ağırlık (kg/g/ton/lb/oz) "
                "birimleri arasında dönüşüm yapar."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "value": {"type": "number", "description": "Dönüştürülecek değer"},
                    "from_unit": {"type": "string", "description": "Kaynak birim"},
                    "to_unit": {"type": "string", "description": "Hedef birim"},
                },
                "required": ["value", "from_unit", "to_unit"],
            },
        },
    },
]

FONKSIYONLAR = {
    "get_weather": get_weather,
    "get_route_distance": get_route_distance,
    "get_exchange_rate": get_exchange_rate,
    "convert_unit": convert_unit,
}

print(f"✓ {len(TOOLS)} araç tanımlı:")
for t in TOOLS:
    fn = t["function"]
    req = ", ".join(fn["parameters"].get("required", []))
    print(f"   - {fn['name']}({req})")

# Şema geçerli mi?
json.dumps(TOOLS)
print("✓ JSON şeması geçerli")

✓ 4 araç tanımlı:
   - get_weather(city)
   - get_route_distance(origin, destination)
   - get_exchange_rate(from_currency, to_currency)
   - convert_unit(value, from_unit, to_unit)
✓ JSON şeması geçerli


---
## 3. Tool Calling Döngüsü

Model araç çağırdıkça sonuçlar mesaj geçmişine `tool` rolüyle eklenir ve
döngü tekrar eder. Böylece bir aracın çıktısı başka bir araca girdi olabilir.


In [6]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_KEY)
MAX_TOOL_TURNS = 5

SISTEM_PROMPTU = """Sen LojistikAI'sın — Cihat Yıldız tarafından geliştirilen, lojistik ve
tedarik zinciri alanında uzmanlaşmış bir asistansın.

Elindeki araçları kullanarak kullanıcının sorularını yanıtla:
- get_weather: hava durumu (sevkiyat koşulları)
- get_route_distance: iki şehir arası karayolu mesafe ve süre
- get_exchange_rate: döviz kuru (navlun maliyeti)
- convert_unit: birim dönüşümü

Kurallar:
- Gerçek veriye ihtiyaç duyduğun her durumda aracı çağır, tahminde bulunma.
- Birden fazla şehir/veri gerekiyorsa araçları paralel çağırabilirsin.
- Bir aracın çıktısı başka bir araca girdi oluyorsa sırayla çağır.
- Yanıtını Türkçe ver, kısa ve net ol.
- Lojistik bağlamı varsa yorumunu ekle (ör. yağış varsa teslimat gecikmesi riski)."""


def _format_arac_cagrisi(ad, argumanlar, sonuc):
    """Araç çağrısını okunabilir formatta döndürür."""
    arg_str = ", ".join(f"{k}={v!r}" for k, v in argumanlar.items())
    sonuc_str = json.dumps(sonuc, ensure_ascii=False)
    return f"   -> {ad}({arg_str})\n   <- {sonuc_str}"


def sohbet(mesaj, gecmis):
    """Tool calling döngüsünü çalıştırır, (yanıt, araç_logu) döndürür."""
    if not os.environ.get("OPENAI_API_KEY"):
        return "⚠ OPENAI_API_KEY tanımlı değil. Space ayarlarından secret ekleyin.", ""

    mesajlar = [{"role": "system", "content": SISTEM_PROMPTU}]

    # Geçmişi ekle (Gradio messages formatı)
    for h in gecmis:
        if isinstance(h, dict) and h.get("role") in ("user", "assistant"):
            mesajlar.append({"role": h["role"], "content": h["content"]})

    mesajlar.append({"role": "user", "content": mesaj})

    log_satirlari = []
    turn = 0

    while turn < MAX_TOOL_TURNS:
        turn += 1

        try:
            yanit = client.chat.completions.create(
                model=MODEL,
                messages=mesajlar,
                tools=TOOLS,
                tool_choice="auto",
                temperature=0.3,
            )
        except Exception as e:
            return f"⚠ Model hatası: {e}", "\n".join(log_satirlari)

        msg = yanit.choices[0].message

        # Araç çağrısı yoksa → nihai yanıt
        if not msg.tool_calls:
            log_satirlari.append(f"\n[Turn {turn}] Nihai Yanıt üretildi.")
            return msg.content, "\n".join(log_satirlari)

        # Araç çağrıları var
        log_satirlari.append(f"\n[Turn {turn}] Araç Çağrıları:")
        mesajlar.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {"name": tc.function.name, "arguments": tc.function.arguments},
                }
                for tc in msg.tool_calls
            ],
        })

        for tc in msg.tool_calls:
            ad = tc.function.name
            try:
                argumanlar = json.loads(tc.function.arguments)
            except json.JSONDecodeError:
                argumanlar = {}

            fn = FONKSIYONLAR.get(ad)
            sonuc = fn(**argumanlar) if fn else {"hata": f"Bilinmeyen araç: {ad}"}

            log_satirlari.append(_format_arac_cagrisi(ad, argumanlar, sonuc))

            mesajlar.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(sonuc, ensure_ascii=False),
            })

    return "⚠ Maksimum araç çağrısı turuna ulaşıldı.", "\n".join(log_satirlari)

print("✓ Tool calling döngüsü hazır")

✓ Tool calling döngüsü hazır


### 3.1 Döngüyü Test Et — Çok Turlu Araç Çağrısı

In [7]:
# Ödevdeki örnek akışın lojistik versiyonu:
# Turn 1 → iki şehrin havası (paralel)
# Turn 2 → sıcaklıkları F'ye çevir (paralel)
# Turn 3 → nihai yanıt

soru = "İstanbul mu daha sıcak Rotterdam mı? Değerleri Fahrenheit olarak da yaz."

yanit, log = sohbet(soru, [])

print("=" * 70)
print(f"Kullanıcı: {soru}")
print("=" * 70)
print(log)
print("\n" + "=" * 70)
print("NİHAİ YANIT:")
print(yanit)

Kullanıcı: İstanbul mu daha sıcak Rotterdam mı? Değerleri Fahrenheit olarak da yaz.

[Turn 1] Araç Çağrıları:
   -> get_weather(city='İstanbul')
   <- {"sehir": "İstanbul", "ulke": "Türkiye Cumhuriyeti", "sicaklik_c": 24.4, "yagis_mm": 0.0, "ruzgar_kmh": 20.9, "durum": "açık"}
   -> get_weather(city='Rotterdam')
   <- {"sehir": "Rotterdam", "ulke": "Hollanda", "sicaklik_c": 33.5, "yagis_mm": 0.0, "ruzgar_kmh": 7.6, "durum": "açık"}

[Turn 2] Araç Çağrıları:
   -> convert_unit(value=24.4, from_unit='C', to_unit='F')
   <- {"deger": 75.92, "birim": "F", "kaynak": "24.4 C"}
   -> convert_unit(value=33.5, from_unit='C', to_unit='F')
   <- {"deger": 92.3, "birim": "F", "kaynak": "33.5 C"}

[Turn 3] Nihai Yanıt üretildi.

NİHAİ YANIT:
İstanbul'un sıcaklığı 24.4°C (75.92°F), Rotterdam'ın sıcaklığı ise 33.5°C (92.3°F) olarak ölçülmüştür. 

Bu durumda Rotterdam İstanbul'dan daha sıcak. Lojistik açısından, yüksek sıcaklıklar özellikle soğuk zincir ürünleri için risk oluşturabilir.


In [8]:
# İkinci test — rota + hava durumu birlikte
soru2 = "Mersin'den Hamburg'a karayolu mesafesi ne kadar? Hamburg'da hava sevkiyata uygun mu?"

yanit2, log2 = sohbet(soru2, [])
print(log2)
print("\nNİHAİ YANIT:")
print(yanit2)


[Turn 1] Araç Çağrıları:
   -> get_route_distance(origin='Mersin', destination='Hamburg')
   <- {"kalkis": "Mersin", "varis": "Hamburg", "mesafe_km": 3420.9, "sure_dakika": 2042, "sure_okunabilir": "34 sa 2 dk"}
   -> get_weather(city='Hamburg')
   <- {"sehir": "Hamburg", "ulke": "Almanya", "sicaklik_c": 31.2, "yagis_mm": 0.0, "ruzgar_kmh": 8.3, "durum": "açık"}

[Turn 2] Nihai Yanıt üretildi.

NİHAİ YANIT:
Mersin'den Hamburg'a karayolu mesafesi yaklaşık 3421 km ve tahmini sürüş süresi 34 saat 2 dakikadır.

Hamburg'da hava durumu açık, sıcaklık 31.2°C ve yağış yok. Bu koşullar sevkiyat için uygundur.


In [9]:
# Üçüncü test — döviz (navlun maliyeti)
soru3 = "2500 EUR navlun bedeli kaç TL eder?"

yanit3, log3 = sohbet(soru3, [])
print(log3)
print("\nNİHAİ YANIT:")
print(yanit3)


[Turn 1] Araç Çağrıları:
   -> get_exchange_rate(from_currency='EUR', to_currency='TRY', amount=2500)
   <- {"kaynak_para": "EUR", "hedef_para": "TRY", "miktar": 2500, "sonuc": 134847, "kur": 53.9388, "tarih": "2026-07-29"}

[Turn 2] Nihai Yanıt üretildi.

NİHAİ YANIT:
2500 EUR navlun bedeli yaklaşık 134,847 TL eder.


---
## 4. Gradio Arayüzünü Colab'da Dene

`share=True` ile geçici public link alırsınız (72 saat geçerli).
Space'e yüklemeden önce arayüzü burada test edebilirsiniz.


In [10]:
import gradio as gr

ORNEK_SORULAR = [
    "İstanbul mu daha sıcak Rotterdam mı? Değerleri Fahrenheit olarak da yaz.",
    "İstanbul'dan Ankara'ya kaç km ve tahmini sürüş süresi ne kadar?",
    "Mersin'den Hamburg'a karayolu mesafesi ne kadar, hava durumu sevkiyata uygun mu?",
    "2500 EUR navlun bedeli kaç TL eder?",
    "İzmir'den Antalya'ya mesafe kaç mil eder?",
    "Bugün Rotterdam limanında hava nasıl, konteyner elleçlemeyi etkiler mi?",
]

# Gradio 5 ve 6 arasındaki API farklarını yönet
GRADIO_MAJOR = int(gr.__version__.split(".")[0])

_chatbot_kwargs = {"label": "Sohbet", "height": 440}
if GRADIO_MAJOR < 6:
    # Gradio 5'te messages formatı için type parametresi gerekli
    _chatbot_kwargs["type"] = "messages"

_blocks_kwargs = {"title": "LojistikAI — Tool Calling"}
_launch_kwargs = {}
if GRADIO_MAJOR < 6:
    _blocks_kwargs["theme"] = gr.themes.Soft()
else:
    # Gradio 6'da tema launch() metoduna taşındı
    _launch_kwargs["theme"] = gr.themes.Soft()


with gr.Blocks(**_blocks_kwargs) as demo:
    gr.Markdown(
        """
        # 🚛 LojistikAI — Tool Calling Demo

        Lojistik operasyonları için **dış veri kaynaklarına (Public API)** bağlanan
        tool calling sistemi. Model, sorunuza göre uygun aracı otomatik seçer;
        arka planda hangi araçları çağırdığı sağ panelde adım adım gösterilir.

        | Araç | Kaynak | Kullanım |
        |------|--------|----------|
        | `get_weather` | Open-Meteo | Sevkiyat koşulları |
        | `get_route_distance` | OSRM | Rota mesafe & süre |
        | `get_exchange_rate` | Frankfurter | Navlun maliyeti |
        | `convert_unit` | Yerel hesap | Birim dönüşümü |
        """
    )

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(**_chatbot_kwargs)
            with gr.Row():
                giris = gr.Textbox(
                    placeholder="Sorunuzu yazın... (ör. İstanbul'dan Ankara'ya kaç km?)",
                    show_label=False,
                    scale=5,
                    container=False,
                )
                gonder = gr.Button("Gönder", variant="primary", scale=1)
            temizle = gr.Button("🗑 Sohbeti temizle", size="sm")

        with gr.Column(scale=2):
            arac_logu = gr.Code(
                label="🔧 Araç Çağrıları (Tool Calls)",
                lines=22,
                interactive=False,
                value="Henüz araç çağrısı yapılmadı.",
            )

    gr.Examples(
        examples=[[s] for s in ORNEK_SORULAR],
        inputs=giris,
        label="Örnek sorular",
    )

    def _yanitla(mesaj, gecmis):
        if not mesaj or not mesaj.strip():
            return gecmis, "", "Boş mesaj gönderildi."

        yanit, log = sohbet(mesaj, gecmis or [])
        yeni_gecmis = (gecmis or []) + [
            {"role": "user", "content": mesaj},
            {"role": "assistant", "content": yanit},
        ]
        return yeni_gecmis, "", (log.strip() or "Bu soruda araç çağrısı yapılmadı.")

    gonder.click(_yanitla, [giris, chatbot], [chatbot, giris, arac_logu])
    giris.submit(_yanitla, [giris, chatbot], [chatbot, giris, arac_logu])
    temizle.click(
        lambda: ([], "", "Henüz araç çağrısı yapılmadı."),
        None,
        [chatbot, giris, arac_logu],
    )

    gr.Markdown(
        """
        ---
        **Geliştirici:** Cihat Yıldız ·
        [Model](https://huggingface.co/cihatyldz/lojistik-lora-adapter) ·
        [Veri seti](https://huggingface.co/datasets/cihatyldz/lojistik-soru-cevap) ·
        [Benchmark](https://huggingface.co/datasets/cihatyldz/lojistik-benchmark)
        """
    )

demo.launch(share=True, debug=False, **_launch_kwargs)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://211b2d99c7bc4121c3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## 5. Space Dosyalarını Oluştur

Yukarıda test ettiğimiz her şeyi tek bir `app.py` dosyasında topluyoruz.


In [11]:
app_kodu = r'''"""
LojistikAI — Tool Calling Demo
================================
Lojistik operasyonları için dış veri kaynaklarıyla (Public API) çalışan
tool calling (function calling) sistemi.

Araçlar:
  1. get_weather        → Open-Meteo   (sevkiyat için hava koşulu)
  2. get_route_distance → OSRM         (iki şehir arası karayolu mesafe/süre)
  3. get_exchange_rate  → Frankfurter  (navlun maliyeti döviz çevrimi)
  4. convert_unit       → yerel hesap  (sıcaklık / mesafe / ağırlık)

Yazar: Cihat Yıldız
"""

import os
import json
import requests
import gradio as gr
from openai import OpenAI

# ----------------------------------------------------------------------
# Yapılandırma
# ----------------------------------------------------------------------
MODEL = "gpt-4o-mini"
API_TIMEOUT = 15
MAX_TOOL_TURNS = 5  # sonsuz döngüye karşı üst sınır

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))


# ======================================================================
# 1. ARAÇ FONKSİYONLARI (Public API'ler)
# ======================================================================

def get_weather(city: str) -> dict:
    """Open-Meteo ile şehrin güncel hava durumunu getirir. API anahtarı gerektirmez."""
    try:
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1, "language": "tr", "format": "json"},
            timeout=API_TIMEOUT,
        ).json()

        if not geo.get("results"):
            return {"hata": f"'{city}' şehri bulunamadı"}

        loc = geo["results"][0]
        lat, lon = loc["latitude"], loc["longitude"]

        wx = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": lat,
                "longitude": lon,
                "current": "temperature_2m,precipitation,wind_speed_10m,weather_code",
                "timezone": "auto",
            },
            timeout=API_TIMEOUT,
        ).json()

        cur = wx["current"]
        kod = cur.get("weather_code", 0)
        durum = _weather_code_tr(kod)

        return {
            "sehir": loc["name"],
            "ulke": loc.get("country", ""),
            "sicaklik_c": cur["temperature_2m"],
            "yagis_mm": cur.get("precipitation", 0),
            "ruzgar_kmh": cur.get("wind_speed_10m", 0),
            "durum": durum,
        }
    except Exception as e:
        return {"hata": f"Hava durumu alınamadı: {e}"}


def get_route_distance(origin: str, destination: str) -> dict:
    """OSRM ile iki şehir arası karayolu mesafesi ve tahmini sürüş süresi."""
    try:
        coords = []
        for sehir in (origin, destination):
            geo = requests.get(
                "https://geocoding-api.open-meteo.com/v1/search",
                params={"name": sehir, "count": 1, "language": "tr", "format": "json"},
                timeout=API_TIMEOUT,
            ).json()
            if not geo.get("results"):
                return {"hata": f"'{sehir}' şehri bulunamadı"}
            r = geo["results"][0]
            coords.append((r["longitude"], r["latitude"], r["name"]))

        (lon1, lat1, ad1), (lon2, lat2, ad2) = coords

        route = requests.get(
            f"https://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}",
            params={"overview": "false"},
            timeout=API_TIMEOUT,
        ).json()

        if route.get("code") != "Ok" or not route.get("routes"):
            return {"hata": "Rota hesaplanamadı (karayolu bağlantısı olmayabilir)"}

        r0 = route["routes"][0]
        km = round(r0["distance"] / 1000, 1)
        dk = round(r0["duration"] / 60)

        return {
            "kalkis": ad1,
            "varis": ad2,
            "mesafe_km": km,
            "sure_dakika": dk,
            "sure_okunabilir": f"{dk // 60} sa {dk % 60} dk",
        }
    except Exception as e:
        return {"hata": f"Rota alınamadı: {e}"}


def get_exchange_rate(from_currency: str, to_currency: str, amount: float = 1.0) -> dict:
    """Frankfurter API ile güncel döviz kuru çevrimi. API anahtarı gerektirmez."""
    try:
        f = from_currency.upper()
        t = to_currency.upper()
        data = requests.get(
            "https://api.frankfurter.app/latest",
            params={"from": f, "to": t, "amount": amount},
            timeout=API_TIMEOUT,
        ).json()

        if "rates" not in data or t not in data["rates"]:
            return {"hata": f"Kur bulunamadı: {f} → {t}"}

        return {
            "kaynak_para": f,
            "hedef_para": t,
            "miktar": amount,
            "sonuc": round(data["rates"][t], 4),
            "kur": round(data["rates"][t] / amount, 6),
            "tarih": data.get("date", ""),
        }
    except Exception as e:
        return {"hata": f"Kur alınamadı: {e}"}


def convert_unit(value: float, from_unit: str, to_unit: str) -> dict:
    """Sıcaklık, mesafe ve ağırlık birimleri arasında dönüşüm yapar (yerel hesap)."""
    f = from_unit.upper().strip()
    t = to_unit.upper().strip()

    # Sıcaklık
    if {f, t} <= {"C", "F", "K"}:
        if f == "C":
            c = value
        elif f == "F":
            c = (value - 32) * 5 / 9
        else:
            c = value - 273.15

        sonuc = c if t == "C" else (c * 9 / 5 + 32 if t == "F" else c + 273.15)
        return {"deger": round(sonuc, 2), "birim": t, "kaynak": f"{value} {f}"}

    # Mesafe
    metre = {"KM": 1000, "M": 1, "MIL": 1609.34, "MI": 1609.34, "NM": 1852, "FT": 0.3048}
    if f in metre and t in metre:
        sonuc = value * metre[f] / metre[t]
        return {"deger": round(sonuc, 3), "birim": t, "kaynak": f"{value} {f}"}

    # Ağırlık
    gram = {"KG": 1000, "G": 1, "TON": 1_000_000, "LB": 453.592, "OZ": 28.3495}
    if f in gram and t in gram:
        sonuc = value * gram[f] / gram[t]
        return {"deger": round(sonuc, 3), "birim": t, "kaynak": f"{value} {f}"}

    return {"hata": f"Desteklenmeyen dönüşüm: {f} → {t}"}


def _weather_code_tr(code: int) -> str:
    """WMO hava kodunu Türkçe açıklamaya çevirir."""
    tablo = {
        0: "açık", 1: "az bulutlu", 2: "parçalı bulutlu", 3: "kapalı",
        45: "sisli", 48: "kırağılı sis",
        51: "hafif çisenti", 53: "çisenti", 55: "yoğun çisenti",
        61: "hafif yağmur", 63: "yağmurlu", 65: "şiddetli yağmur",
        71: "hafif kar", 73: "karlı", 75: "yoğun kar",
        80: "sağanak", 81: "kuvvetli sağanak", 82: "şiddetli sağanak",
        95: "gök gürültülü fırtına", 96: "dolulu fırtına",
    }
    return tablo.get(code, "bilinmiyor")


# ======================================================================
# 2. TOOL (FUNCTION) TANIMLARI — JSON Şeması
# ======================================================================

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": (
                "Belirtilen şehrin güncel hava durumunu getirir. Sevkiyat planlaması, "
                "rota güvenliği ve soğuk zincir kararları için kullanılır."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Şehir adı, örn: 'İstanbul', 'Hamburg', 'Rotterdam'",
                    }
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_route_distance",
            "description": (
                "İki şehir arasındaki karayolu mesafesini (km) ve tahmini sürüş süresini "
                "hesaplar. Nakliye maliyeti, teslim süresi ve rota planlaması için kullanılır."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {"type": "string", "description": "Kalkış şehri"},
                    "destination": {"type": "string", "description": "Varış şehri"},
                },
                "required": ["origin", "destination"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_exchange_rate",
            "description": (
                "Güncel döviz kuru ile para birimi çevrimi yapar. Navlun fiyatlandırma, "
                "ithalat/ihracat maliyet hesaplama için kullanılır."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "from_currency": {
                        "type": "string",
                        "description": "Kaynak para birimi kodu, örn: 'USD', 'EUR', 'TRY'",
                    },
                    "to_currency": {
                        "type": "string",
                        "description": "Hedef para birimi kodu, örn: 'TRY', 'USD'",
                    },
                    "amount": {
                        "type": "number",
                        "description": "Çevrilecek miktar (varsayılan 1)",
                    },
                },
                "required": ["from_currency", "to_currency"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "convert_unit",
            "description": (
                "Sıcaklık (C/F/K), mesafe (km/m/mil/nm/ft) ve ağırlık (kg/g/ton/lb/oz) "
                "birimleri arasında dönüşüm yapar."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "value": {"type": "number", "description": "Dönüştürülecek değer"},
                    "from_unit": {"type": "string", "description": "Kaynak birim"},
                    "to_unit": {"type": "string", "description": "Hedef birim"},
                },
                "required": ["value", "from_unit", "to_unit"],
            },
        },
    },
]

FONKSIYONLAR = {
    "get_weather": get_weather,
    "get_route_distance": get_route_distance,
    "get_exchange_rate": get_exchange_rate,
    "convert_unit": convert_unit,
}


# ======================================================================
# 3. TOOL CALLING DÖNGÜSÜ
# ======================================================================

SISTEM_PROMPTU = """Sen LojistikAI'sın — Cihat Yıldız tarafından geliştirilen, lojistik ve
tedarik zinciri alanında uzmanlaşmış bir asistansın.

Elindeki araçları kullanarak kullanıcının sorularını yanıtla:
- get_weather: hava durumu (sevkiyat koşulları)
- get_route_distance: iki şehir arası karayolu mesafe ve süre
- get_exchange_rate: döviz kuru (navlun maliyeti)
- convert_unit: birim dönüşümü

Kurallar:
- Gerçek veriye ihtiyaç duyduğun her durumda aracı çağır, tahminde bulunma.
- Birden fazla şehir/veri gerekiyorsa araçları paralel çağırabilirsin.
- Bir aracın çıktısı başka bir araca girdi oluyorsa sırayla çağır.
- Yanıtını Türkçe ver, kısa ve net ol.
- Lojistik bağlamı varsa yorumunu ekle (ör. yağış varsa teslimat gecikmesi riski)."""


def _format_arac_cagrisi(ad, argumanlar, sonuc):
    """Araç çağrısını okunabilir formatta döndürür."""
    arg_str = ", ".join(f"{k}={v!r}" for k, v in argumanlar.items())
    sonuc_str = json.dumps(sonuc, ensure_ascii=False)
    return f"   -> {ad}({arg_str})\n   <- {sonuc_str}"


def sohbet(mesaj, gecmis):
    """Tool calling döngüsünü çalıştırır, (yanıt, araç_logu) döndürür."""
    if not os.environ.get("OPENAI_API_KEY"):
        return "⚠ OPENAI_API_KEY tanımlı değil. Space ayarlarından secret ekleyin.", ""

    mesajlar = [{"role": "system", "content": SISTEM_PROMPTU}]

    # Geçmişi ekle (Gradio messages formatı)
    for h in gecmis:
        if isinstance(h, dict) and h.get("role") in ("user", "assistant"):
            mesajlar.append({"role": h["role"], "content": h["content"]})

    mesajlar.append({"role": "user", "content": mesaj})

    log_satirlari = []
    turn = 0

    while turn < MAX_TOOL_TURNS:
        turn += 1

        try:
            yanit = client.chat.completions.create(
                model=MODEL,
                messages=mesajlar,
                tools=TOOLS,
                tool_choice="auto",
                temperature=0.3,
            )
        except Exception as e:
            return f"⚠ Model hatası: {e}", "\n".join(log_satirlari)

        msg = yanit.choices[0].message

        # Araç çağrısı yoksa → nihai yanıt
        if not msg.tool_calls:
            log_satirlari.append(f"\n[Turn {turn}] Nihai Yanıt üretildi.")
            return msg.content, "\n".join(log_satirlari)

        # Araç çağrıları var
        log_satirlari.append(f"\n[Turn {turn}] Araç Çağrıları:")
        mesajlar.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {"name": tc.function.name, "arguments": tc.function.arguments},
                }
                for tc in msg.tool_calls
            ],
        })

        for tc in msg.tool_calls:
            ad = tc.function.name
            try:
                argumanlar = json.loads(tc.function.arguments)
            except json.JSONDecodeError:
                argumanlar = {}

            fn = FONKSIYONLAR.get(ad)
            sonuc = fn(**argumanlar) if fn else {"hata": f"Bilinmeyen araç: {ad}"}

            log_satirlari.append(_format_arac_cagrisi(ad, argumanlar, sonuc))

            mesajlar.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(sonuc, ensure_ascii=False),
            })

    return "⚠ Maksimum araç çağrısı turuna ulaşıldı.", "\n".join(log_satirlari)


# ======================================================================
# 4. GRADIO ARAYÜZÜ
# ======================================================================

ORNEK_SORULAR = [
    "İstanbul mu daha sıcak Rotterdam mı? Değerleri Fahrenheit olarak da yaz.",
    "İstanbul'dan Ankara'ya kaç km ve tahmini sürüş süresi ne kadar?",
    "Mersin'den Hamburg'a karayolu mesafesi ne kadar, hava durumu sevkiyata uygun mu?",
    "2500 EUR navlun bedeli kaç TL eder?",
    "İzmir'den Antalya'ya mesafe kaç mil eder?",
    "Bugün Rotterdam limanında hava nasıl, konteyner elleçlemeyi etkiler mi?",
]

# Gradio 5 ve 6 arasındaki API farklarını yönet
GRADIO_MAJOR = int(gr.__version__.split(".")[0])

_chatbot_kwargs = {"label": "Sohbet", "height": 440}
if GRADIO_MAJOR < 6:
    # Gradio 5'te messages formatı için type parametresi gerekli
    _chatbot_kwargs["type"] = "messages"

_blocks_kwargs = {"title": "LojistikAI — Tool Calling"}
_launch_kwargs = {}
if GRADIO_MAJOR < 6:
    _blocks_kwargs["theme"] = gr.themes.Soft()
else:
    # Gradio 6'da tema launch() metoduna taşındı
    _launch_kwargs["theme"] = gr.themes.Soft()


with gr.Blocks(**_blocks_kwargs) as demo:
    gr.Markdown(
        """
        # 🚛 LojistikAI — Tool Calling Demo

        Lojistik operasyonları için **dış veri kaynaklarına (Public API)** bağlanan
        tool calling sistemi. Model, sorunuza göre uygun aracı otomatik seçer;
        arka planda hangi araçları çağırdığı sağ panelde adım adım gösterilir.

        | Araç | Kaynak | Kullanım |
        |------|--------|----------|
        | `get_weather` | Open-Meteo | Sevkiyat koşulları |
        | `get_route_distance` | OSRM | Rota mesafe & süre |
        | `get_exchange_rate` | Frankfurter | Navlun maliyeti |
        | `convert_unit` | Yerel hesap | Birim dönüşümü |
        """
    )

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(**_chatbot_kwargs)
            with gr.Row():
                giris = gr.Textbox(
                    placeholder="Sorunuzu yazın... (ör. İstanbul'dan Ankara'ya kaç km?)",
                    show_label=False,
                    scale=5,
                    container=False,
                )
                gonder = gr.Button("Gönder", variant="primary", scale=1)
            temizle = gr.Button("🗑 Sohbeti temizle", size="sm")

        with gr.Column(scale=2):
            arac_logu = gr.Code(
                label="🔧 Araç Çağrıları (Tool Calls)",
                lines=22,
                interactive=False,
                value="Henüz araç çağrısı yapılmadı.",
            )

    gr.Examples(
        examples=[[s] for s in ORNEK_SORULAR],
        inputs=giris,
        label="Örnek sorular",
    )

    def _yanitla(mesaj, gecmis):
        if not mesaj or not mesaj.strip():
            return gecmis, "", "Boş mesaj gönderildi."

        yanit, log = sohbet(mesaj, gecmis or [])
        yeni_gecmis = (gecmis or []) + [
            {"role": "user", "content": mesaj},
            {"role": "assistant", "content": yanit},
        ]
        return yeni_gecmis, "", (log.strip() or "Bu soruda araç çağrısı yapılmadı.")

    gonder.click(_yanitla, [giris, chatbot], [chatbot, giris, arac_logu])
    giris.submit(_yanitla, [giris, chatbot], [chatbot, giris, arac_logu])
    temizle.click(
        lambda: ([], "", "Henüz araç çağrısı yapılmadı."),
        None,
        [chatbot, giris, arac_logu],
    )

    gr.Markdown(
        """
        ---
        **Geliştirici:** Cihat Yıldız ·
        [Model](https://huggingface.co/cihatyldz/lojistik-lora-adapter) ·
        [Veri seti](https://huggingface.co/datasets/cihatyldz/lojistik-soru-cevap) ·
        [Benchmark](https://huggingface.co/datasets/cihatyldz/lojistik-benchmark)
        """
    )


if __name__ == "__main__":
    demo.launch(**_launch_kwargs)
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_kodu)

print(f"✓ app.py yazıldı ({len(app_kodu):,} karakter)")

import ast
ast.parse(app_kodu)
print("✓ Syntax geçerli")

✓ app.py yazıldı (17,583 karakter)
✓ Syntax geçerli


In [12]:
requirements = """gradio>=5.0.0
openai>=1.50.0
requests>=2.31.0
"""

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements)

print("✓ requirements.txt yazıldı")
print(requirements)

✓ requirements.txt yazıldı
gradio>=5.0.0
openai>=1.50.0
requests>=2.31.0



In [13]:
readme = r'''---
title: LojistikAI Tool Calling
emoji: 🚛
colorFrom: blue
colorTo: indigo
sdk: gradio
sdk_version: 5.49.1
app_file: app.py
pinned: false
license: mit
short_description: Lojistik için public API'lerle çalışan tool calling asistanı
---

# 🚛 LojistikAI — Tool Calling Demo

Lojistik ve tedarik zinciri sorularını **canlı dış veri kaynaklarına (Public API)** bağlanarak yanıtlayan tool calling (function calling) sistemi.

Model, kullanıcının sorusuna göre uygun aracı otomatik seçer, gerekirse birden fazla aracı sırayla veya paralel çağırır ve **hangi araçları hangi parametrelerle çağırdığını arayüzde adım adım gösterir**.

## 🔧 Araçlar

| Araç | Veri Kaynağı | API Anahtarı | Lojistik Kullanımı |
|------|-------------|:---:|--------------------|
| `get_weather` | [Open-Meteo](https://open-meteo.com/) | ❌ | Sevkiyat koşulları, soğuk zincir, rota güvenliği |
| `get_route_distance` | [OSRM](https://project-osrm.org/) | ❌ | Karayolu mesafe & süre, nakliye maliyeti |
| `get_exchange_rate` | [Frankfurter](https://frankfurter.app/) | ❌ | Navlun fiyatlandırma, ithalat/ihracat maliyeti |
| `convert_unit` | Yerel hesaplama | — | Sıcaklık (C/F/K), mesafe (km/mil/nm), ağırlık (kg/ton/lb) |

Dört API de tamamen ücretsiz ve anahtarsızdır.

## 💬 Örnek Çalışma Akışı

**Kullanıcı:** *"İstanbul mu daha sıcak Rotterdam mı? Değerleri Fahrenheit olarak da yaz."*

```text
[Turn 1] Araç Çağrıları:
   -> get_weather(city='İstanbul')
   <- {"sehir": "İstanbul", "sicaklik_c": 27.4, "durum": "az bulutlu", "yagis_mm": 0}
   -> get_weather(city='Rotterdam')
   <- {"sehir": "Rotterdam", "sicaklik_c": 18.1, "durum": "yağmurlu", "yagis_mm": 1.2}

[Turn 2] Araç Çağrıları:
   -> convert_unit(value=27.4, from_unit='C', to_unit='F')
   <- {"deger": 81.32, "birim": "F", "kaynak": "27.4 C"}
   -> convert_unit(value=18.1, from_unit='C', to_unit='F')
   <- {"deger": 64.58, "birim": "F", "kaynak": "18.1 C"}

[Turn 3] Nihai Yanıt:
İstanbul 27.4°C (81.3°F) ile Rotterdam'dan (18.1°C / 64.6°F) daha sıcak.
Rotterdam'da yağış var — liman elleçlemesinde gecikme riski göz önünde bulundurulmalı.
```

Bu akış arayüzün sağ panelinde canlı olarak görüntülenir.

## 🏗️ Mimari

```
Kullanıcı sorusu
      │
      ▼
┌─────────────────────────────────────┐
│  gpt-4o-mini + TOOLS (JSON şeması)  │
└──────────────┬──────────────────────┘
               │ tool_calls var mı?
        ┌──────┴───────┐
       Evet            Hayır
        │                │
        ▼                ▼
  Fonksiyonu       Nihai yanıt
  çalıştır         (döngü biter)
        │
        ▼
  Public API çağrısı
        │
        ▼
  Sonucu mesaj geçmişine
  tool rolüyle ekle
        │
        └──► döngüye geri dön (max 5 tur)
```

Çok turlu döngü sayesinde bir aracın çıktısı başka bir araca girdi olabilir (örn. hava durumundan gelen °C değeri `convert_unit`'e beslenir).

## 🚀 Kurulum

### Hugging Face Spaces

1. Yeni bir Space oluşturun (SDK: **Gradio**)
2. `app.py`, `requirements.txt` ve `README.md` dosyalarını yükleyin
3. **Settings → Variables and secrets** bölümünden `OPENAI_API_KEY` secret'ını ekleyin
4. Space otomatik olarak derlenip yayına alınır

### Yerel çalıştırma

```bash
git clone <repo-url>
cd lojistik-tool-calling
pip install -r requirements.txt

export OPENAI_API_KEY="sk-..."
python app.py
```

Arayüz `http://localhost:7860` adresinde açılır.

## 📁 Dosyalar

```
├── app.py            # Gradio arayüzü + araç fonksiyonları + tool calling döngüsü
├── requirements.txt  # gradio, openai, requests
└── README.md         # Bu dosya (HF Spaces frontmatter dahil)
```

## ⚙️ Teknik Detaylar

- **Model:** `gpt-4o-mini` (native function calling desteği)
- **Tool tanımı:** OpenAI JSON Schema formatı — her araç için `name`, `description` ve tipli `parameters`
- **Paralel çağrı:** Model tek turda birden fazla aracı çağırabilir (örn. iki şehrin hava durumu)
- **Zincirleme çağrı:** `MAX_TOOL_TURNS = 5` ile sınırlı çok turlu döngü
- **Hata yönetimi:** API hataları araç sonucunda `{"hata": "..."}` olarak döner, model bunu kullanıcıya açıklar
- **Zaman aşımı:** Her API çağrısı için 15 saniye

## 🔗 İlgili Çalışmalar

| Kaynak | Link |
|--------|------|
| Fine-tuned model | [`cihatyldz/lojistik-lora-adapter`](https://huggingface.co/cihatyldz/lojistik-lora-adapter) |
| Eğitim veri seti | [`cihatyldz/lojistik-soru-cevap`](https://huggingface.co/datasets/cihatyldz/lojistik-soru-cevap) |
| BPE tokenizer | [`cihatyldz/lojistik-bpe-tokenizer`](https://huggingface.co/cihatyldz/lojistik-bpe-tokenizer) |
| Özel benchmark | [`cihatyldz/lojistik-benchmark`](https://huggingface.co/datasets/cihatyldz/lojistik-benchmark) |

## 👤 Yazar

**Cihat Yıldız** — Kıdemli Veri Bilimcisi, Lojistik Sektörü
[HuggingFace](https://huggingface.co/cihatyldz)

## 📄 Lisans

MIT
'''

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)

print(f"✓ README.md yazıldı ({len(readme):,} karakter)")

✓ README.md yazıldı (4,731 karakter)


---
## 6. Hugging Face Spaces'e Yayınla

Space oluşturulur, dosyalar yüklenir ve `OPENAI_API_KEY` secret olarak eklenir.

> **Donanım:** CPU Basic yeterli — bu uygulama GPU kullanmaz (sadece API çağrıları yapar).


In [15]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)

print("Mevcut Space'leriniz:\n")
for s in api.list_spaces(author=HF_USERNAME):
    try:
        rt = api.get_space_runtime(s.id)
        print(f"  {s.id:<50} {rt.hardware or 'bilinmiyor':<15} {rt.stage}")
    except Exception as e:
        print(f"  {s.id:<50} (durum alınamadı)")

Mevcut Space'leriniz:

  cihatyldz/lojistik-llm-annotation-demo             bilinmiyor      SLEEPING
  cihatyldz/cv-matching-engine                       cpu-basic       RUNNING
  cihatyldz/akce-banka-ai                            zero-a10g       RUNNING
  cihatyldz/kervan-lojistik-rag                      zero-a10g       RUNNING
  cihatyldz/sifahane-turkish-medical                 bilinmiyor      SLEEPING
  cihatyldz/mizan-fact-checker                       cpu-basic       RUNNING
  cihatyldz/carsi-ecommerce-ner                      cpu-basic       RUNNING


In [17]:
from huggingface_hub import create_repo

try:
    create_repo(
        repo_id=SPACE_ID,
        repo_type="space",
        space_sdk="gradio",
        space_hardware="zero-a10g",   # ← ücretsiz ZeroGPU slotu
        private=False,
        exist_ok=True,
    )
    print(f"✓ Space hazır (ZeroGPU): {SPACE_ID}")
except Exception as e:
    print(f"⚠ {e}")

✓ Space hazır (ZeroGPU): cihatyldz/lojistik-tool-calling


In [18]:
# ===== 2. Dosyaları yükle =====
for dosya in ["app.py", "requirements.txt", "README.md"]:
    api.upload_file(
        path_or_fileobj=dosya,
        path_in_repo=dosya,
        repo_id=SPACE_ID,
        repo_type="space",
    )
    print(f"  ✓ {dosya} yüklendi")

print(f"\n✓ Tüm dosyalar yüklendi")

  ✓ app.py yüklendi
  ✓ requirements.txt yüklendi
  ✓ README.md yüklendi

✓ Tüm dosyalar yüklendi


In [19]:
# ===== 3. OPENAI_API_KEY secret'ını ekle =====
# Bu olmadan Space açılır ama model çağrısı yapamaz.
try:
    api.add_space_secret(
        repo_id=SPACE_ID,
        key="OPENAI_API_KEY",
        value=OPENAI_KEY,
    )
    print("✓ OPENAI_API_KEY secret eklendi")
except Exception as e:
    print(f"⚠ Secret eklenemedi: {e}")
    print("  Manuel ekleyin: Space → Settings → Variables and secrets → New secret")

✓ OPENAI_API_KEY secret eklendi


In [20]:
# ===== 4. Durumu kontrol et =====
import time

print("Space derleniyor, 30 saniye bekleniyor...\n")
time.sleep(30)

try:
    runtime = api.get_space_runtime(SPACE_ID)
    print(f"Durum   : {runtime.stage}")
    print(f"Donanım : {runtime.hardware}")
except Exception as e:
    print(f"Durum alınamadı: {e}")

print(f"\n{'='*60}")
print(f"  🚀 CANLI DEMO")
print(f"  https://huggingface.co/spaces/{SPACE_ID}")
print(f"{'='*60}")
print("\nNot: İlk derleme 2-4 dakika sürebilir.")
print("Space sayfasında 'Building' → 'Running' olmasını bekleyin.")

Space derleniyor, 30 saniye bekleniyor...

Durum   : BUILDING
Donanım : None

  🚀 CANLI DEMO
  https://huggingface.co/spaces/cihatyldz/lojistik-tool-calling

Not: İlk derleme 2-4 dakika sürebilir.
Space sayfasında 'Building' → 'Running' olmasını bekleyin.


---
## ✅ Teslim Kontrol Listesi

| Gereksinim | Durum |
|-----------|-------|
| Public API seçimi (Open-Meteo, OSRM, Frankfurter) | ☐ |
| Tool Call tasarımı (JSON Şeması — 4 araç) | ☐ |
| Model entegrasyonu (gpt-4o-mini function calling) | ☐ |
| Araç adımlarının kullanıcıya gösterilmesi | ☐ |
| Gradio arayüzü | ☐ |
| Hugging Face Spaces'te yayın | ☐ |
| `app.py` + `requirements.txt` + `README.md` | ☐ |

### Teslim edilecekler

| Öğe | Link |
|-----|------|
| Canlı demo | `https://huggingface.co/spaces/cihatyldz/lojistik-tool-calling` |
| Kod deposu | Aynı Space (Files sekmesi) veya GitHub |

### Sorun giderme

**Space "Build error" veriyor** → Files → `requirements.txt` sürümlerini kontrol edin, Logs sekmesinden hatayı okuyun.

**"OPENAI_API_KEY tanımlı değil" uyarısı** → Settings → Variables and secrets → New secret ekleyin, sonra Restart Space.

**Araç çağrısı boş dönüyor** → İlgili public API geçici olarak yanıt vermiyor olabilir; `API_TIMEOUT` değerini artırmayı deneyin.
